In [1]:
import pandas as pd
from pathlib import Path

In [2]:
RUTA = Path(
    r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\01_raw\elecciones"
)

In [3]:
for carpeta in RUTA.iterdir():
    print(carpeta.name)

2015
2016
2019_04
2019_11
2023


In [4]:
archivos = []

for carpeta in RUTA.iterdir():
    
    archivo_partidos = list(carpeta.glob("03*.DAT"))[0]
    archivo_votos = list(carpeta.glob("08*.DAT"))[0]
    
    archivos.append({
        "año": carpeta.name,
        "partidos": archivo_partidos,
        "votos": archivo_votos
    })

pd.DataFrame(archivos)

,año,partidos,votos
0,2015,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...
1,2016,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...
2,2019_04,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...
3,2019_11,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...
4,2023,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...,C:\Users\herre\OneDrive\Desktop\Proyectos DATA...


In [5]:
prueba = [x for x in archivos if x["año"] == "2019_11"][0]

prueba

{'año': '2019_11',
 'partidos': WindowsPath('C:/Users/herre/OneDrive/Desktop/Proyectos DATA/SPRINT 13/01_data/01_raw/elecciones/2019_11/03021911.DAT'),
 'votos': WindowsPath('C:/Users/herre/OneDrive/Desktop/Proyectos DATA/SPRINT 13/01_data/01_raw/elecciones/2019_11/08021911.DAT')}

Leemos las primeras lineas de partidos

In [6]:
with open(prueba['partidos'], encoding='latin1') as f:
    for i in range(5):
        print(f.readline())

02201911000002AHORA CANARIAS                                    AHORA CANARIAS: Alternativa Nacionalista Canaria (ANC) y Unidad del Pueblo                                                                            000002000002000002

02201911000003ANDECHA                                           ANDECHA ASTUR                                                                                                                                         000003000003000003

02201911000005AUNACV                                            AUNA COMUNITAT VALENCIANA                                                                                                                             000005000005000005

02201911000006AVANT ADELANTE LOS VERDES                         AVANT ADELANTE LOS VERDES                                                                                                                             000006000006000006

02201911000007AVANT ADELANTE LOS VERDES                         

Leemos las primeras lineas de votos

In [7]:
with open(prueba['votos'], encoding='latin1') as f:
    for i in range(5):
        print(f.readline())

022019111011190000830011108900002

022019111012390000980000041900000

022019111012990000440001411200000

022019111014190000580000084200000

022019111019990000550004748500000



Como los archivos son tipo .DAT y no tienen separadores, le decimos que lea de tal carácter a otro  y es una columna, otro rango de carácteres es otra columna y así hasta finalizar. 

In [8]:
partidos = pd.read_fwf(
    prueba['partidos'],
    colspecs=[
        (0,2),    # tipo elección
        (2,6),    # año
        (6,8),    # mes
        (8,14),   # código candidatura
        (14,64),  # siglas
        (64,214)  # nombre
    ],
    header=None,
    encoding='latin1'
)

partidos.columns = [
    'tipo',
    'año',
    'mes',
    'cod_partido',
    'siglas',
    'nombre'
]

partidos.head()

,tipo,año,mes,cod_partido,siglas,nombre
0,2,2019,11,2,AHORA CANARIAS,AHORA CANARIAS: Alternativa Nacionalista Canar...
1,2,2019,11,3,ANDECHA,ANDECHA ASTUR
2,2,2019,11,5,AUNACV,AUNA COMUNITAT VALENCIANA
3,2,2019,11,6,AVANT ADELANTE LOS VERDES,AVANT ADELANTE LOS VERDES
4,2,2019,11,7,AVANT ADELANTE LOS VERDES,LOS VERDES ECOPACIFISTAS ADELANTE


Hacemos lo mismo que antes pero con la tabla 'votos'.

In [9]:
votos = pd.read_fwf(
    prueba['votos'],
    colspecs=[
        (0,2),    # tipo
        (2,6),    # año
        (6,8),    # mes
        (8,9),    # vuelta
        (9,11),   # ccaa
        (11,13),  # provincia
        (13,14),  # distrito
        (14,20),  # cod_partido
        (20,28),  # votos
        (28,33)   # diputados
    ],
    header=None,
    encoding='latin1'
)

votos.columns = [
    'tipo',
    'año',
    'mes',
    'vuelta',
    'ccaa',
    'provincia',
    'distrito',
    'cod_partido',
    'votos',
    'diputados'
]

votos.head()

,tipo,año,mes,vuelta,ccaa,provincia,distrito,cod_partido,votos,diputados
0,2,2019,11,1,1,11,9,83,111089,2
1,2,2019,11,1,1,23,9,98,419,0
2,2,2019,11,1,1,29,9,44,14112,0
3,2,2019,11,1,1,41,9,58,842,0
4,2,2019,11,1,1,99,9,55,47485,0


Unimos ambas tablas, partidos + votos de 2019

In [10]:
elecciones = votos.merge(
    partidos[['cod_partido', 'siglas', 'nombre']],
    on='cod_partido',
    how='left'
)

elecciones.head()



,tipo,año,mes,vuelta,ccaa,provincia,distrito,cod_partido,votos,diputados,siglas,nombre
0,2,2019,11,1,1,11,9,83,111089,2,PP,PARTIDO POPULAR
1,2,2019,11,1,1,23,9,98,419,0,PUM+J,POR UN MUNDO MÁS JUSTO
2,2,2019,11,1,1,29,9,44,14112,0,MÁS PAÍS-AN,MÁS PAÍS-ANDALUCÍA
3,2,2019,11,1,1,41,9,58,842,0,PCPA,PARTIDO COMUNISTA DEL PUEBLO ANDALUZ
4,2,2019,11,1,1,99,9,55,47485,0,PACMA,PARTIDO ANIMALISTA CONTRA EL MALTRATO ANIMAL


Ahora calculamos el total por Comunidades Autónomas

In [11]:
elecciones_ccaa = elecciones[elecciones['provincia'] == 99].copy()

elecciones_ccaa = elecciones_ccaa[
    ['año', 'mes', 'ccaa', 'siglas', 'nombre', 'votos', 'diputados']
]

elecciones_ccaa.head()

,año,mes,ccaa,siglas,nombre,votos,diputados
4,2019,11,1,PACMA,PARTIDO ANIMALISTA CONTRA EL MALTRATO ANIMAL,47485,0
10,2019,11,4,PODEMOS-EUIB,UNIDAS PODEMOS-UNIDES PODEM,82225,2
13,2019,11,5,CCa-PNC-NC,COALICIÓN CANARIA-NUEVA CANARIAS,124289,2
14,2019,11,5,PDSJE,PARTIDO DEMÓCRATA SOCIAL JUBILADOS EUROPEOS,713,0
26,2019,11,8,PPSO,PLATAFORMA DEL PUEBLO SORIANO,1466,0


In [12]:
elecciones_ccaa = elecciones_ccaa[elecciones_ccaa['ccaa'] != 99]

Creamos columna de Comunidad a partir del diccionario mapa_ccaa, dónde cáda clave correspoinde a una CCAA

In [13]:
mapa_ccaa = {
    1:'Andalucía', 2:'Aragón', 3:'Asturias', 4:'Baleares',
    5:'Canarias', 6:'Cantabria', 7:'Castilla-La Mancha',
    8:'Castilla y León', 9:'Cataluña', 10:'Extremadura',
    11:'Galicia', 12:'Madrid', 13:'Navarra', 14:'País Vasco',
    15:'Murcia', 16:'La Rioja', 17:'Comunidad Valenciana',
    18:'Ceuta', 19:'Melilla'
}

elecciones_ccaa['comunidad'] = elecciones_ccaa['ccaa'].map(mapa_ccaa)

elecciones_ccaa.head()

,año,mes,ccaa,siglas,nombre,votos,diputados,comunidad
4,2019,11,1,PACMA,PARTIDO ANIMALISTA CONTRA EL MALTRATO ANIMAL,47485,0,Andalucía
10,2019,11,4,PODEMOS-EUIB,UNIDAS PODEMOS-UNIDES PODEM,82225,2,Baleares
13,2019,11,5,CCa-PNC-NC,COALICIÓN CANARIA-NUEVA CANARIAS,124289,2,Canarias
14,2019,11,5,PDSJE,PARTIDO DEMÓCRATA SOCIAL JUBILADOS EUROPEOS,713,0,Canarias
26,2019,11,8,PPSO,PLATAFORMA DEL PUEBLO SORIANO,1466,0,Castilla y León


In [14]:
elecciones_ccaa.to_csv(
    r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\02_processed\elecciones\elecciones_2019_11.csv",
    index=False,
    encoding="utf-8"
)

Aplicamos la programación modular, se divide el proceso en en funciones independiente.

El algoritmo hace:
- Lectura de ficheros
- Integración de datos
- Filtrado territorial
- Enriquecimiento de variables
- Exportación de resultados

In [15]:
MAPA_CCAA = {
    1:'Andalucía', 2:'Aragón', 3:'Asturias', 4:'Baleares',
    5:'Canarias', 6:'Cantabria', 7:'Castilla-La Mancha',
    8:'Castilla y León', 9:'Cataluña', 10:'Extremadura',
    11:'Galicia', 12:'Madrid', 13:'Navarra', 14:'País Vasco',
    15:'Murcia', 16:'La Rioja', 17:'Comunidad Valenciana',
    18:'Ceuta', 19:'Melilla'
}

In [16]:
def leer_partidos(ruta_partidos):
    partidos = pd.read_fwf(
        ruta_partidos,
        colspecs=[(0,2), (2,6), (6,8), (8,14), (14,64), (64,214)],
        header=None,
        encoding='latin1'
    )
    partidos.columns = ['tipo', 'año', 'mes', 'cod_partido', 'siglas', 'nombre']
    return partidos

In [17]:
def leer_votos(ruta_votos):
    votos = pd.read_fwf(
        ruta_votos,
        colspecs=[(0,2), (2,6), (6,8), (8,9), (9,11), (11,13), (13,14), (14,20), (20,28), (28,33)],
        header=None,
        encoding='latin1'
    )
    votos.columns = ['tipo', 'año', 'mes', 'vuelta', 'ccaa', 'provincia', 'distrito', 'cod_partido', 'votos', 'diputados']
    return votos

In [18]:
def unir_partidos_votos(votos, partidos):
    return votos.merge(
        partidos[['cod_partido', 'siglas', 'nombre']],
        on='cod_partido',
        how='left'
    )

In [19]:
def filtrar_ccaa(df):
    df = df[df['provincia'] == 99].copy()
    df = df[df['ccaa'] != 99].copy()
    return df

In [20]:
def agregar_nombre_ccaa(df):
    df['comunidad'] = df['ccaa'].map(MAPA_CCAA)
    return df

In [21]:
def procesar_eleccion(carpeta):
    ruta_partidos = list(carpeta.glob("03*.DAT"))[0]
    ruta_votos = list(carpeta.glob("08*.DAT"))[0]

    partidos = leer_partidos(ruta_partidos)
    votos = leer_votos(ruta_votos)

    df = unir_partidos_votos(votos, partidos)
    df = filtrar_ccaa(df)
    df = agregar_nombre_ccaa(df)

    return df[['año', 'mes', 'ccaa', 'comunidad', 'siglas', 'nombre', 'votos', 'diputados']]

In [22]:
def main():
    ruta_raw = Path(
        r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\01_raw\elecciones"
    )

    ruta_salida = Path(
        r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\02_processed\elecciones"
    )

    resultados = []

    for carpeta in ruta_raw.iterdir():
        if carpeta.is_dir():
            df = procesar_eleccion(carpeta)
            resultados.append(df)

            nombre_archivo = f"elecciones_{carpeta.name}.csv"
            df.to_csv(
                ruta_salida / nombre_archivo,
                index=False,
                encoding="utf-8"
            )

    elecciones_total = pd.concat(resultados, ignore_index=True)
    elecciones_total.to_csv(
        ruta_salida / "elecciones_total.csv",
        index=False,
        encoding="utf-8"
    )

    return elecciones_total

In [23]:
elecciones[['año','mes']].drop_duplicates().sort_values(['año','mes'])

,año,mes
0,2019,11


In [24]:
elecciones.head()

,tipo,año,mes,vuelta,ccaa,provincia,distrito,cod_partido,votos,diputados,siglas,nombre
0,2,2019,11,1,1,11,9,83,111089,2,PP,PARTIDO POPULAR
1,2,2019,11,1,1,23,9,98,419,0,PUM+J,POR UN MUNDO MÁS JUSTO
2,2,2019,11,1,1,29,9,44,14112,0,MÁS PAÍS-AN,MÁS PAÍS-ANDALUCÍA
3,2,2019,11,1,1,41,9,58,842,0,PCPA,PARTIDO COMUNISTA DEL PUEBLO ANDALUZ
4,2,2019,11,1,1,99,9,55,47485,0,PACMA,PARTIDO ANIMALISTA CONTRA EL MALTRATO ANIMAL
